In [1]:
pip install numpy pandas tqdm torch datasets transformers peft bitsandbytes accelerate

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import gc
import json
import math
import time
import random
import hashlib
import inspect
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# IMPORTANT:
# Use plain tqdm, not tqdm.auto, to avoid notebook widget rendering issues.
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset, Dataset, DatasetDict
from datasets.utils.logging import disable_progress_bar

# Disable HF datasets widget/progress rendering in notebooks.
disable_progress_bar()

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    AdaLoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [ ]:
from huggingface_hub import login # or set it manually
login(token=os.environ.get("HF_TOKEN", ""))

In [5]:
LORAGA_AVAILABLE = False
_estimate_gradient = None
_LoraGAContext = None
_LoraGAConfig = None
_save_loraga_model_init = None
_save_loraga_model_final = None

try:
    from peft import LoraGAConfig as _lgc
    from peft.utils.lora_ga_utils import (
        estimate_gradient as _eg,
        LoraGAContext as _lgctx,
        save_loraga_model_init as _slgi,
        save_loraga_model_final as _slgf,
    )
    _LoraGAConfig = _lgc
    _estimate_gradient = _eg
    _LoraGAContext = _lgctx
    _save_loraga_model_init = _slgi
    _save_loraga_model_final = _slgf
    LORAGA_AVAILABLE = True
    print("[INFO] LoRA-GA custom PEFT detected — loraga method available.")
except ImportError:
    print(
        "[WARN] LoRA-GA custom PEFT not found.\n"
        "       Install: cd LoRA-GA && pip install -e peft\n"
        "       Method 'loraga' will raise RuntimeError if selected."
    )

[INFO] LoRA-GA custom PEFT detected — loraga method available.


In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DTYPE_MAP = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}

print(f"[INFO] Device: {DEVICE}")

[INFO] Device: cuda


In [39]:
from dataclasses import dataclass, asdict, field
from typing import Any, Dict, List, Optional, Tuple

@dataclass
class RunConfig:
    # -----------------------------
    # Model
    # -----------------------------
    model_name: str = "google/gemma-2-2b-it"
    use_4bit: bool = True
    torch_dtype: str = "bfloat16"
    gradient_checkpointing: bool = True

    # -----------------------------
    # Train track
    # -----------------------------
    # one of: gsm8k, math, alpaca_gpt4
    train_track: str = "gsm8k"

    # -----------------------------
    # Run mode
    # -----------------------------
    # smoke  -> quick sanity check
    # screen -> Day 1 ranking
    # final  -> full eval for finalists
    mode: str = "smoke"

    # IMPORTANT:
    # Leave this empty/None unless you intentionally want to override.
    # The eval dispatcher will fill sane defaults based on train_track + mode.
    eval_buckets: Optional[List[str]] = None

    # -----------------------------
    # Baseline
    # -----------------------------
    # full_ft, lora, bitfit, topk_full, automode, adalora, dora, loraga
    method: str = "lora"

    # -----------------------------
    # Optimization
    # -----------------------------
    seed: int = 8
    epochs: int = 2
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    train_batch_size: int = 1
    eval_batch_size: int = 1
    grad_accum_steps: int = 16
    max_grad_norm: float = 1.0
    max_train_samples: Optional[int] = None
    max_eval_samples: Optional[int] = None

    # -----------------------------
    # Tokenization / generation
    # -----------------------------
    max_source_len: int = 1024
    max_target_len: int = 512
    max_new_tokens: int = 256
    num_beams: int = 1
    do_sample: bool = True
    temperature: float = 0.7
    top_p: float = 0.95
    sampling_k: int = 5   # used for GSM8K majority voting

    # -----------------------------
    # Standard LoRA / DoRA
    # -----------------------------
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: Tuple[str, ...] = ("q_proj", "v_proj")

    # -----------------------------
    # AdaLoRA
    # -----------------------------
    adalora_init_r: int = 16
    adalora_target_r: int = 8
    adalora_tinit: int = 200
    adalora_tfinal: int = 1000
    adalora_deltaT: int = 10
    adalora_total_step: Optional[int] = None

    # -----------------------------
    # Top-k full FT
    # -----------------------------
    topk_k: int = 6
    topk_strategy: str = "last_k"   # last_k or deep_block
    deep_block_start: int = 13
    deep_block_end: int = 19

    # -----------------------------
    # AutoMode / Dynamic Full FT
    # -----------------------------
    dynamic_updates: int = 10
    dynamic_threshold: int = 10
    dynamic_log_every: int = 1

    # -----------------------------
    # LoRA-GA
    # -----------------------------
    loraga_bs: int = 4
    loraga_steps_for_grad_est: int = 4

    # -----------------------------
    # Logging
    # -----------------------------
    output_root: str = "./runs_gemma2_2b"
    save_model: bool = True
    save_every_eval_dump: bool = True

    # -----------------------------
    # External / optional
    # -----------------------------
    mtbench_judge_model: Optional[str] = None
    openai_api_key_env: str = "OPENAI_API_KEY"


cfg = RunConfig()
print(cfg)

RunConfig(model_name='google/gemma-2-2b-it', use_4bit=True, torch_dtype='bfloat16', gradient_checkpointing=True, train_track='gsm8k', mode='smoke', eval_buckets=None, method='lora', seed=8, epochs=2, learning_rate=5e-05, weight_decay=0.01, warmup_ratio=0.03, train_batch_size=1, eval_batch_size=1, grad_accum_steps=16, max_grad_norm=1.0, max_train_samples=None, max_eval_samples=None, max_source_len=1024, max_target_len=512, max_new_tokens=256, num_beams=1, do_sample=True, temperature=0.7, top_p=0.95, sampling_k=5, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_target_modules=('q_proj', 'v_proj'), adalora_init_r=16, adalora_target_r=8, adalora_tinit=200, adalora_tfinal=1000, adalora_deltaT=10, adalora_total_step=None, topk_k=6, topk_strategy='last_k', deep_block_start=13, deep_block_end=19, dynamic_updates=10, dynamic_threshold=10, dynamic_log_every=1, loraga_bs=4, loraga_steps_for_grad_est=4, output_root='./runs_gemma2_2b', save_model=True, save_every_eval_dump=True, mtbench_judge_mod

In [8]:
def set_seed(seed: int):
    """Reproducibility: sets Python, NumPy, and PyTorch seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def timestamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S")


def make_run_id(cfg: RunConfig) -> str:
    """Deterministic 12-char hash of the config — used as folder name."""
    payload = {
        k: v for k, v in asdict(cfg).items()
        if k not in ("output_root", "save_model", "eval_buckets",
                     "mtbench_questions_path")
    }
    return hashlib.md5(
        json.dumps(payload, sort_keys=True).encode()
    ).hexdigest()[:12]


def ensure_dirs(cfg: RunConfig, run_id: str) -> Dict[str, Path]:
    """Create output directory tree and return path dict."""
    root = Path(cfg.output_root)
    paths = {
        "root": root,
        "run": root / run_id,
        "configs": root / run_id / "configs",
        "logs": root / run_id / "logs",
        "evals": root / run_id / "evals",
        "dynamic": root / run_id / "dynamic",
        "checkpoints": root / run_id / "checkpoints",
        "preds": root / run_id / "preds",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def save_json(obj: Any, path: Path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def get_layer_index_from_name(name: str) -> Optional[int]:
    """
    Extract the transformer layer index from a parameter name.
    For Gemma/LLaMA:  model.layers.{N}.self_attn.q_proj.weight → N
    Returns None for non-layer params (embed_tokens, lm_head, etc.).
    """
    m = re.search(r"model\.layers\.(\d+)", name)
    return int(m.group(1)) if m else None


def get_all_transformer_layer_ids(model: nn.Module) -> List[int]:
    """Return sorted list of all transformer layer indices in the model."""
    ids = set()
    for name, _ in model.named_parameters():
        idx = get_layer_index_from_name(name)
        if idx is not None:
            ids.add(idx)
    return sorted(ids)


def count_trainable_params(model: nn.Module) -> Tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def append_result_row(
    cfg: RunConfig, run_id: str,
    metrics: Dict[str, Any],
    train_time_sec: float,
    peak_vram_gb: float,
):
    """Append one result row to the master CSV."""
    row = {
        "run_id": run_id,
        "model": cfg.model_name,
        "train_track": cfg.train_track,
        "method": cfg.method,
        "seed": cfg.seed,
        "lr": cfg.learning_rate,
        "epochs": cfg.epochs,
        "lora_r": cfg.lora_r,
        "lora_alpha": cfg.lora_alpha,
        "dynamic_updates": cfg.dynamic_updates,
        "dynamic_threshold": cfg.dynamic_threshold,
        "train_time_sec": round(train_time_sec, 1),
        "peak_vram_gb": round(peak_vram_gb, 2),
        **metrics,
    }
    csv_path = Path(cfg.output_root) / "master_runs.csv"
    df = pd.DataFrame([row])
    df.to_csv(csv_path, mode="a", header=not csv_path.exists(), index=False)

In [9]:
def build_math_prompt(question: str) -> str:
    """
    Prompt template for GSM8K and MATH.
    We ask for step-by-step reasoning followed by the answer in #### format.
    This format is used consistently for BOTH training and evaluation so the
    model sees the same structure in both phases.
    """
    return (
        "Solve the following math problem step by step. "
        "End your response with the final answer in the format '#### <answer>'.\n\n"
        f"Question: {question}\n\nAnswer:"
    )


def build_instruction_prompt(
    tokenizer,
    instruction: str,
    input_text: Optional[str] = None
) -> str:
    """
    Prompt for Alpaca-GPT4 (instruction-following track).
    Uses the model's chat template if available (preferred for -it models).
    Falls back to a simple format for base models.
    """
    user_content = f"Instruction: {instruction}"
    if input_text and input_text.strip():
        user_content += f"\n\nInput: {input_text}"
    messages = [{"role": "user", "content": user_content}]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        # Base models typically lack a chat template
        return user_content + "\n\nResponse:"


def build_mmlu_prompt(
    question: str,
    choices: List[str],
    fewshot_examples: List[Dict[str, Any]]
) -> str:
    """
    5-shot MMLU prompt. Few-shot examples come from the dev split of each
    subject, so they never overlap with the test questions.
    """
    letters = ["A", "B", "C", "D"]
    header = "You are taking a multiple-choice exam. Reply with only the correct option letter.\n\n"
    shots = []
    for ex in fewshot_examples:
        s = ex["question"] + "\n"
        for i, c in enumerate(ex["choices"]):
            s += f"{letters[i]}. {c}\n"
        s += f"Answer: {letters[ex['answer']]}\n\n"
        shots.append(s)
    cur = question + "\n"
    for i, c in enumerate(choices):
        cur += f"{letters[i]}. {c}\n"
    cur += "Answer:"
    return header + "".join(shots) + cur


def build_arc_prompt(question: str, choices: List[str]) -> str:
    """Prompt for ARC-Challenge (zero-shot)."""
    letters = ["A", "B", "C", "D", "E"]
    prompt = "Answer the science multiple-choice question. Reply with only the correct option letter.\n\n"
    prompt += question + "\n"
    for i, c in enumerate(choices):
        prompt += f"{letters[i]}. {c}\n"
    prompt += "Answer:"
    return prompt

In [10]:
def get_tokenizer(model_name: str) -> AutoTokenizer:
    """
    Load the tokenizer. Key settings:
    - pad_token = eos_token when no pad token exists (common for Gemma/LLaMA).
    - padding_side = "right" is correct for training (causal LM with right-pad).
      For batched generation you'd want left-padding, but we generate one
      example at a time in this notebook so it doesn't matter.
    """
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok

In [ ]:
def load_gsm8k_dataset() -> DatasetDict:
    return load_dataset("openai/gsm8k", "main")


def load_math_dataset() -> DatasetDict:
    """
    More defensive MATH loader.
    Your earlier version failed on hendrycks/competition_math in this environment.
    Try multiple candidates and use the first one that works.
    """
    candidates = [
        ("DigitalLearningGmbH/MATH-lighteval", None),
        ("lighteval/MATH", None),
        ("competition_math", None),
        ("hendrycks/competition_math", None),
    ]

    last_err = None
    for name, subset in candidates:
        try:
            print(f"[INFO] Trying MATH dataset: {name}")
            if subset is None:
                return load_dataset(name)
            return load_dataset(name, subset)
        except Exception as e:
            last_err = e
            print(f"[WARN] Failed to load {name}: {e}")

    raise RuntimeError(f"Could not load any MATH dataset candidate. Last error: {last_err}")


def load_alpaca_gpt4_dataset() -> DatasetDict:
    return load_dataset("vicgalle/alpaca-gpt4")


def load_mmlu_dataset() -> DatasetDict:
    return load_dataset("cais/mmlu", "all")


def load_arc_dataset() -> DatasetDict:
    return load_dataset("allenai/ai2_arc", "ARC-Challenge")


def normalize_gsm8k(ds: DatasetDict) -> DatasetDict:
    def _map(ex):
        return {
            "prompt": build_math_prompt(ex["question"]),
            "target": ex["answer"],
            "raw_question": ex["question"],
            "raw_answer": ex["answer"],
        }

    out = DatasetDict()
    for split in ds:
        out[split] = ds[split].map(_map)
    return out


def normalize_math(ds: DatasetDict) -> DatasetDict:
    possible_train = "train" if "train" in ds else list(ds.keys())[0]
    possible_test = "test" if "test" in ds else ("validation" if "validation" in ds else list(ds.keys())[-1])

    def _map(ex):
        problem = ex.get("problem", ex.get("question", ""))
        solution = ex.get("solution", ex.get("answer", ""))
        return {
            "prompt": build_math_prompt(problem),
            "target": solution,
            "raw_question": problem,
            "raw_answer": solution,
        }

    out = DatasetDict()
    out["train"] = ds[possible_train].map(_map)
    out["test"] = ds[possible_test].map(_map)
    return out


def normalize_alpaca(ds: DatasetDict) -> DatasetDict:
    train_key = "train" if "train" in ds else list(ds.keys())[0]

    def _map(ex):
        prompt = build_instruction_prompt(ex["instruction"], ex.get("input", ""))
        return {
            "prompt": prompt,
            "target": ex["output"],
            "raw_instruction": ex["instruction"],
            "raw_input": ex.get("input", ""),
            "raw_output": ex["output"],
        }

    out = DatasetDict()
    out["train"] = ds[train_key].map(_map)
    return out


def load_training_track(cfg: RunConfig) -> DatasetDict:
    if cfg.train_track == "gsm8k":
        return normalize_gsm8k(load_gsm8k_dataset())
    elif cfg.train_track == "math":
        return normalize_math(load_math_dataset())
    elif cfg.train_track == "alpaca_gpt4":
        return normalize_alpaca(load_alpaca_gpt4_dataset())
    else:
        raise ValueError(f"Unknown train_track: {cfg.train_track}")


train_ds = load_training_track(cfg)
print(train_ds)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'prompt', 'target', 'raw_question', 'raw_answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer', 'prompt', 'target', 'raw_question', 'raw_answer'],
        num_rows: 1319
    })
})


: 

In [12]:
# Each normaliser produces a DatasetDict with a "train" and "test" split,
# where each example has "prompt", "target", "raw_question", "raw_answer".

def normalize_gsm8k(ds: DatasetDict, tokenizer) -> DatasetDict:
    def _map(ex):
        return {
            "prompt": build_math_prompt(ex["question"]),
            "target": ex["answer"],
            "raw_question": ex["question"],
            "raw_answer": ex["answer"],
        }
    return DatasetDict({split: ds[split].map(_map) for split in ds})


def normalize_metamathqa(ds: DatasetDict) -> DatasetDict:
    """
    MetaMathQA format: {"query": ..., "response": ..., "type": ...}
    We keep training only (no built-in test split — evaluate on GSM8K/MATH).
    """
    train_key = "train" if "train" in ds else list(ds.keys())[0]
    def _map(ex):
        return {
            "prompt": build_math_prompt(ex["query"]),
            "target": ex["response"],
            "raw_question": ex["query"],
            "raw_answer": ex["response"],
        }
    return DatasetDict({"train": ds[train_key].map(_map)})


def normalize_alpaca(ds: DatasetDict, tokenizer) -> DatasetDict:
    train_key = "train" if "train" in ds else list(ds.keys())[0]
    def _map(ex):
        return {
            "prompt": build_instruction_prompt(
                tokenizer, ex["instruction"], ex.get("input", "")
            ),
            "target": ex["output"],
            "raw_instruction": ex["instruction"],
            "raw_input": ex.get("input", ""),
            "raw_output": ex["output"],
        }
    return DatasetDict({"train": ds[train_key].map(_map)})


def normalize_math_eval(ds: DatasetDict) -> DatasetDict:
    """Normalise the MATH benchmark for evaluation only."""
    possible_test = "test" if "test" in ds else list(ds.keys())[-1]
    def _map(ex):
        problem = ex.get("problem", ex.get("question", ""))
        solution = ex.get("solution", ex.get("answer", ""))
        return {
            "prompt": build_math_prompt(problem),
            "target": solution,
            "raw_question": problem,
            "raw_answer": solution,
        }
    return DatasetDict({"test": ds[possible_test].map(_map)})


def load_training_track(cfg: RunConfig, tokenizer) -> DatasetDict:
    if cfg.train_track == "gsm8k":
        return normalize_gsm8k(load_gsm8k_dataset(), tokenizer)
    elif cfg.train_track == "metamathqa":
        return normalize_metamathqa(load_metamathqa_dataset())
    elif cfg.train_track == "alpaca_gpt4":
        return normalize_alpaca(load_alpaca_gpt4_dataset(), tokenizer)
    else:
        raise ValueError(f"Unknown train_track: {cfg.train_track!r}")

In [13]:
def tokenize_sft_example(
    prompt: str,
    target: str,
    tokenizer,
    max_source_len: int,
    max_target_len: int,
) -> Dict[str, List[int]]:
    """
    Tokenises one prompt+target pair for causal LM SFT.

    FIX-2: We ONLY compute loss on the target (response) tokens.
    The prompt tokens are masked with -100 in the labels tensor so the
    cross-entropy loss ignores them.

    Why this matters:
    - Without masking, the model wastes gradient steps re-predicting the known
      prompt (e.g., "Solve the following math problem step by step…").
    - Different methods differ in how many parameters affect the prompt
      representation; unmasked prompts create an unfair comparison.
    - This is the standard recipe in TRL's SFTTrainer and is how LISA,
      AdaLoRA, and DoRA all train their models.

    Implementation note:
    We tokenise prompt and full sequence separately (add_special_tokens=False
    for prompt) to find the exact token boundary. The special token at the
    very start belongs to the prompt side → mask it.
    """
    max_total = max_source_len + max_target_len

    # Tokenise the full sequence (prompt + space + target) with truncation
    full_text = prompt + " " + target
    full_enc = tokenizer(
        full_text,
        max_length=max_total,
        truncation=True,
        add_special_tokens=True,
    )
    full_ids = full_enc["input_ids"]

    # Tokenise just the prompt to determine where the response begins.
    # We do NOT add special tokens here — they were already counted in full_enc.
    prompt_enc = tokenizer(
        prompt,
        max_length=max_source_len,
        truncation=True,
        add_special_tokens=False,
    )
    # +1 for the BOS token that the full encode inserts at position 0
    prompt_len = len(prompt_enc["input_ids"]) + 1
    prompt_len = min(prompt_len, len(full_ids))  # safety clamp

    # Labels: -100 masks the prompt, target tokens are kept for loss
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    return {
        "input_ids": full_ids,
        "attention_mask": full_enc["attention_mask"],
        "labels": labels,
    }


def build_train_dataset(ds: Dataset, tokenizer, cfg: RunConfig) -> Dataset:
    if cfg.max_train_samples is not None:
        ds = ds.select(range(min(cfg.max_train_samples, len(ds))))

    def _tok(batch):
        result = {"input_ids": [], "attention_mask": [], "labels": []}
        for p, t in zip(batch["prompt"], batch["target"]):
            enc = tokenize_sft_example(
                p, t, tokenizer, cfg.max_source_len, cfg.max_target_len
            )
            result["input_ids"].append(enc["input_ids"])
            result["attention_mask"].append(enc["attention_mask"])
            result["labels"].append(enc["labels"])
        return result

    ds = ds.map(_tok, batched=True, remove_columns=ds.column_names)
    return ds

In [14]:
class CausalLMCollator:
    """
    FIX-3: Correctly handles label padding.

    When sequences in a batch have different lengths, we right-pad them to the
    longest sequence in the batch. Padding positions in the *labels* must be
    set to -100 so PyTorch's cross-entropy loss ignores them.

    The original notebook did:
        labels[labels == pad_token_id] = -100    ← WRONG
    because the collator used tokenizer.pad() on the labels directly.
    Padding tokens in labels should ALWAYS be -100, not pad_token_id.
    """
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Separate labels before padding (tokenizer.pad() doesn't know about -100)
        label_list = [f.pop("labels") for f in features]

        # Pad input_ids and attention_mask
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt",
        )

        # Pad labels to the same length with -100
        max_len = batch["input_ids"].shape[1]
        padded_labels = []
        for lab in label_list:
            pad_len = max_len - len(lab)
            padded_labels.append(lab + [-100] * pad_len)
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)

        return batch

In [15]:
def load_base_model(cfg: RunConfig) -> nn.Module:
    """
    Load the pretrained model with the correct dtype and device mapping.

    FIX-7: After enabling gradient checkpointing, we call
           model.enable_input_require_grads(). This is required because
           gradient checkpointing recomputes activations during backward,
           and PEFT hooks that insert gradient computation into the
           computation graph need the first module's input to have
           requires_grad=True. Without this call, LoRA and DoRA adapters
           can silently receive zero gradients when gradient_checkpointing=True.

    FIX-8: device_map={"": DEVICE} forces all layers to one device.
           device_map="auto" uses HuggingFace's dispatch framework which
           adds forward hooks that conflict with PEFT's hooks, especially
           under gradient checkpointing. Single-GPU setups should always
           use {"": device_str}.

    FIX-4: full_ft is incompatible with 4-bit quantization.
           bitsandbytes stores weights as int4 tensors that have no gradient
           support. We override use_4bit=False for full_ft automatically.
    """
    TORCH_DTYPE = DTYPE_MAP[cfg.torch_dtype]
    force_no_4bit = (cfg.method == "full_ft")

    model_kwargs = {
        "torch_dtype": TORCH_DTYPE,
        "device_map": {"": DEVICE},   # FIX-8: explicit single-device mapping
    }

    if cfg.use_4bit and not force_no_4bit:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=TORCH_DTYPE,
        )
        print("[INFO] Loading in 4-bit NF4 (QLoRA-style)")
    elif force_no_4bit and cfg.use_4bit:
        print("[INFO] full_ft: overriding use_4bit=False (FIX-4)")

    model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **model_kwargs)

    if cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False        # cache incompatible with grad ckpt
        model.enable_input_require_grads()    # FIX-7: required for PEFT grads

    if cfg.use_4bit and not force_no_4bit:
        model = prepare_model_for_kbit_training(model)

    return model

In [16]:
def apply_full_ft(model: nn.Module) -> nn.Module:
    """
    All parameters trainable. This is the accuracy upper bound baseline.
    On Gemma-2-2B (2.24B params) this requires ~18 GB VRAM for bfloat16
    weights + gradients + optimizer states (Adam stores 2 momentum tensors
    per parameter).
    """
    for p in model.parameters():
        p.requires_grad = True
    return model

In [17]:
def apply_lora(model: nn.Module, cfg: RunConfig) -> nn.Module:
    """
    Standard LoRA with frozen base weights.
    Only q_proj and v_proj receive trainable low-rank adapters (the primary
    config from the research runs). The base model weights are never updated.

    Parameter count for r=16, Gemma-2-2B, q+v targets:
        2 × 26 layers × 2 × (2048 × 16 + 16 × 2048) ≈ 8.9M trainable params
        = 0.40% of 2.24B total.
    """
    peft_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_target_modules),
        bias="none",
        task_type="CAUSAL_LM",
    )
    return get_peft_model(model, peft_cfg)

In [18]:
def apply_dora(model: nn.Module, cfg: RunConfig) -> nn.Module:
    """
    DoRA (Weight-Decomposed Low-Rank Adaptation, ICML 2024 Oral).
    Decomposes the pretrained weight W into magnitude m and direction V̂,
    then applies LoRA ONLY to the direction component.

    Two design notes:
    1. DoRA typically needs a slightly LOWER lr than LoRA (0.8× is a good
       starting point) because the magnitude vector is an additional degree of
       freedom that can amplify updates. We keep the same lr here for a fair
       comparison; revisit if DoRA underperforms.
    2. DoRA adds a scalar magnitude parameter per output dimension per target
       module, so trainable param count is slightly higher than LoRA.

    Requires PEFT ≥ 0.11.0. Check: 'use_dora' in inspect.signature(LoraConfig).
    """
    try:
        import inspect
        if "use_dora" not in inspect.signature(LoraConfig.__init__).parameters:
            raise ImportError("PEFT version does not support use_dora.")
    except Exception as e:
        raise RuntimeError(
            f"DoRA requires PEFT >= 0.11.0. Update with: pip install -U peft\n{e}"
        )

    peft_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_target_modules),
        bias="none",
        task_type="CAUSAL_LM",
        use_dora=True,   # the only change from standard LoRA
    )
    return get_peft_model(model, peft_cfg)

In [19]:
def apply_adalora(model: nn.Module, cfg: RunConfig, total_steps: int) -> nn.Module:
    """
    AdaLoRA (ICLR 2023) — adaptive rank allocation using SVD.
    Parameterises updates as P·Λ·Q^T and prunes small singular values.

    FIX-9: We use init_r=32, target_r=16 to give AdaLoRA the same final
    capacity as LoRA r=16. The original notebook used target_r=8, giving
    AdaLoRA half the rank budget — an unfair comparison that would make
    AdaLoRA look weaker than it actually is.

    The budget schedule:
    - tinit steps: no pruning, let the model warm up
    - tinit → (total - tfinal): linear cubic decay from init_r to target_r
    - last tfinal steps: frozen at target_r

    IMPORTANT: you MUST call model.update_and_allocate(global_step) after
    every optimizer step. This is handled in the training loop below.
    """
    peft_cfg = AdaLoraConfig(
        init_r=cfg.adalora_init_r,          # starting rank (before pruning)
        target_r=cfg.adalora_target_r,      # final rank budget per matrix
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_target_modules),
        task_type="CAUSAL_LM",
        bias="none",
        total_step=total_steps,
        tinit=cfg.adalora_tinit,
        tfinal=cfg.adalora_tfinal,
        deltaT=cfg.adalora_deltaT,
    )
    return get_peft_model(model, peft_cfg)

In [20]:
def apply_loraga(model: nn.Module, cfg: RunConfig, train_loader: DataLoader) -> nn.Module:
    """
    LoRA-GA (NeurIPS 2024) — gradient-aligned LoRA initialisation.
    Instead of Kaiming random initialisation, LoRA-GA initialises A and B
    using the top-r singular vectors of the full gradient matrix at step 0.
    This aligns ΔBA with the true gradient direction, giving convergence speed
    comparable to full fine-tuning from the very first step.

    FIX-5: Uses the CORRECT official API from Outsider565/LoRA-GA:
        Step 1: estimate_gradient() — one or more forward+backward passes on
                a small batch to compute the gradient of each weight matrix.
        Step 2: LoraGAContext — attaches the gradient dict to the model as
                model.named_grad so that LoraGAModel can read it during init.
        Step 3: get_peft_model() — wraps the model with LoRA-GA adapters.

    The gradient estimation takes ~2-5 minutes for a 2B model but only
    happens once at the start of each run.

    Requires: cd LoRA-GA && pip install -e peft
    """
    if not LORAGA_AVAILABLE:
        raise RuntimeError(
            "LoRA-GA custom PEFT not installed.\n"
            "Install: git clone https://github.com/Outsider565/LoRA-GA.git\n"
            "         cd LoRA-GA && pip install -r requirements.txt && pip install -e peft"
        )

    peft_cfg = _LoraGAConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_target_modules),
        bias="none",
        task_type="CAUSAL_LM",
        # LoraGAConfig-specific fields:
        bsz=cfg.loraga_bsz,       # batch size during gradient estimation
        iters=cfg.loraga_iters,   # how many batches to average over
    )

    print(f"[LoRA-GA] Estimating gradients over {cfg.loraga_iters} batches …")
    # estimate_gradient does forward+backward on small batches and stores
    # the accumulated gradient per weight matrix name → gradient dict
    from accelerate import Accelerator
    accelerator = Accelerator()
    named_grad = _estimate_gradient(
        model=model,
        dataloader=train_loader,
        accelerator=accelerator,
        quant_flag=cfg.use_4bit,   # True if model was loaded in 4-bit
    )
    print("[LoRA-GA] Gradient estimation complete. Initialising adapters …")

    # LoraGAContext temporarily attaches named_grad to model so that
    # get_peft_model can read it when initialising LoRA weight matrices
    with _LoraGAContext(model=model, named_grad=named_grad):
        model = get_peft_model(model=model, peft_config=peft_cfg)

    # Zero out any leftover gradients from the estimation phase
    model.zero_grad()
    print("[LoRA-GA] Model ready.")
    return model

In [21]:
def apply_topk_deep_block(model: nn.Module, cfg: RunConfig) -> nn.Module:
    """
    Static selective full FT: only the specified deep block of transformer
    layers is trainable. Everything else (including lm_head and embed_tokens)
    is frozen.

    For Gemma-2-2B (26 layers, 0-indexed 0-25):
      deep_block_start=13, deep_block_end=19 → layers 13-19 (7 layers)

    This is the primary "static alternative" baseline that motivates dynamic
    allocation: the heatmap from the 9B runs shows important layers cluster
    in the deep region but don't form a perfectly fixed block.
    """
    all_ids = get_all_transformer_layer_ids(model)

    if cfg.topk_strategy == "deep_block":
        selected = {
            i for i in all_ids
            if cfg.deep_block_start <= i <= cfg.deep_block_end
        }
    elif cfg.topk_strategy == "last_k":
        selected = set(all_ids[-cfg.topk_k:])
    else:
        raise ValueError(f"Unknown topk_strategy: {cfg.topk_strategy!r}")

    for name, p in model.named_parameters():
        idx = get_layer_index_from_name(name)
        p.requires_grad = (idx in selected)

    return model

In [22]:
class AutoModeGradAccumulator:
    """
    FIX-1: Accumulates squared gradient norms across ALL micro-steps within
    a gradient accumulation window.

    Original problem: The previous notebook called collect_layer_grad_scores()
    ONLY once per optimizer step (inside the if grad_accum == 0 block). This
    meant it only saw gradients from the single most recent micro-step
    backward pass — 1 out of 16 micro-steps were used, giving a much noisier
    importance signal.

    Correct behaviour (matching the 9B research notebook):
    - Call accumulate() after EVERY loss.backward() call.
    - Call get_scores_and_clear() only when you want to make a switching
      decision (i.e., every update_interval optimizer steps).

    The score for layer L is the RMS gradient magnitude:
        score(L) = sqrt( Σ_params(||∇||²) / Σ_params(n_elements) )
    This normalises by parameter count so large layers don't dominate just
    because they have more parameters.
    """
    def __init__(self):
        self._sum_sq: Dict[int, float] = {}   # layer_idx → accumulated ∑ ||∇||²
        self._counts: Dict[int, int] = {}     # layer_idx → accumulated parameter count

    def accumulate(self, model: nn.Module):
        """Call after every loss.backward(), before optim.zero_grad()."""
        for name, p in model.named_parameters():
            if p.grad is None:
                continue
            idx = get_layer_index_from_name(name)
            if idx is None:
                continue
            sq = float(torch.sum(p.grad.detach().float() ** 2).item())
            cnt = p.numel()
            self._sum_sq[idx] = self._sum_sq.get(idx, 0.0) + sq
            self._counts[idx] = self._counts.get(idx, 0) + cnt

    def get_scores_and_clear(self) -> Dict[int, float]:
        """
        Compute per-layer RMS gradient scores and reset the accumulator.
        Call this once per switching decision point.
        """
        scores = {}
        for idx in self._sum_sq:
            cnt = max(self._counts[idx], 1)
            scores[idx] = math.sqrt(self._sum_sq[idx] / cnt)
        self._sum_sq.clear()
        self._counts.clear()
        return scores

In [23]:
def set_layer_mode(model: nn.Module, layer_idx: int, full_ft: bool):
    """
    Switch one transformer layer between full-FT and LoRA-only mode.

    full_ft=True:
      - Enable requires_grad on all base weight params in the layer.
      - Disable requires_grad on LoRA adapter params (they've been merged
        conceptually; in this implementation we don't do the merge-unmerge
        dance to keep it simple — the base weights are the trainable entity).

    full_ft=False (LoRA mode):
      - Disable requires_grad on base weights.
      - Enable requires_grad on LoRA adapter params (lora_A, lora_B).
      - DoRA magnitude vectors (magnitude_vector) also stay trainable.

    Design note: Unlike the research notebook, we do NOT call module.merge()
    and module.reset_lora_parameters() here. The merge/reset approach is more
    faithful to the theory (merged weight = θ + (α/r)·BA) but requires careful
    optimizer state management. This simplified version achieves the same
    selective trainability structure and is easier to validate.
    """
    prefix = f"model.layers.{layer_idx}."
    ADAPTER_TAGS = ("lora_A", "lora_B", "lora_embedding_A",
                    "lora_embedding_B", "magnitude_vector")

    for name, p in model.named_parameters():
        if not name.startswith(prefix):
            continue

        is_adapter = any(tag in name for tag in ADAPTER_TAGS)

        if full_ft:
            # Full-FT mode: train the base weight directly, freeze adapters
            p.requires_grad = not is_adapter
        else:
            # LoRA mode: freeze base weights, train only adapter params
            p.requires_grad = is_adapter


def rebuild_optimizer_and_scheduler(
    model: nn.Module,
    cfg: RunConfig,
    total_steps: int,
    current_step: int,
) -> Tuple:
    """
    Rebuild optimizer and scheduler after a layer mode change.

    Why rebuild:
    - The set of trainable parameters changed → old optimizer has stale
      param groups that may include now-frozen params or miss new ones.
    - We use AdamW (not AdamW8bit) for simplicity here; swap to AdamW8bit
      from bitsandbytes if you need to save optimizer state memory.

    The new scheduler starts from the remaining steps (total - current).
    This preserves the overall cosine decay shape.
    """
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable,
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    remaining_steps = max(1, total_steps - current_step)
    warmup_steps = max(1, int(remaining_steps * cfg.warmup_ratio))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=remaining_steps,
    )
    return optimizer, scheduler

In [24]:
def apply_automode(model: nn.Module, cfg: RunConfig) -> nn.Module:
    """
    AutoMode initial state: identical to standard LoRA.
    The key difference happens at runtime — the training loop periodically
    promotes high-gradient-norm layers to full-FT mode.
    """
    return apply_lora(model, cfg)


def apply_dyn_full_only(model: nn.Module, cfg: RunConfig) -> nn.Module:
    """
    dyn_full_only: same gradient-norm controller as AutoMode but non-selected
    layers are completely FROZEN (no LoRA fallback) instead of being kept in
    LoRA mode. This is the critical ablation that isolates whether the LoRA
    channel in AutoMode is doing real work.

    If AutoMode >> dyn_full_only: the LoRA fallback is essential.
    If AutoMode ≈ dyn_full_only: the benefit comes purely from gradient
    selection, not from the hybrid LoRA/FFT architecture.

    Since dyn_full_only starts without LoRA adapters, we begin from the base
    model (fully frozen) and dynamically enable layers as training progresses.
    """
    # Start fully frozen; switching logic will enable layers progressively
    for p in model.parameters():
        p.requires_grad = False
    return model


def set_dyn_full_only_layer(model: nn.Module, layer_idx: int, full_ft: bool):
    """
    For dyn_full_only: switch is binary — train full layer or freeze entirely.
    No LoRA adapters exist, so all params in the layer are base weights.
    """
    prefix = f"model.layers.{layer_idx}."
    for name, p in model.named_parameters():
        if name.startswith(prefix):
            p.requires_grad = full_ft

In [25]:
def build_model_for_method(
    cfg: RunConfig,
    tokenizer,
    train_loader: DataLoader,
) -> nn.Module:
    """
    Single entry point: load base model + apply the chosen method.
    Returns the model ready for training.
    """
    model = load_base_model(cfg)

    if cfg.method == "full_ft":
        model = apply_full_ft(model)

    elif cfg.method == "lora":
        model = apply_lora(model, cfg)

    elif cfg.method == "dora":
        model = apply_dora(model, cfg)

    elif cfg.method == "adalora":
        # total_steps not known yet at model-build time; pass a dummy value.
        # apply_adalora is called again inside train_model() with the real count.
        model = apply_adalora(model, cfg, total_steps=1000)

    elif cfg.method == "loraga":
        model = apply_loraga(model, cfg, train_loader)

    elif cfg.method == "topk_deep_block":
        model = apply_topk_deep_block(model, cfg)

    elif cfg.method == "automode":
        model = apply_automode(model, cfg)

    elif cfg.method == "dyn_full_only":
        model = apply_dyn_full_only(model, cfg)

    else:
        raise ValueError(f"Unknown method: {cfg.method!r}")

    trainable, total = count_trainable_params(model)
    print(
        f"[{cfg.method}] Trainable: {trainable:,} / {total:,} "
        f"({100 * trainable / total:.3f}%)"
    )
    return model

In [26]:
def extract_hash_answer(text: str) -> Optional[str]:
    """Extract the final answer after '#### ' (GSM8K convention)."""
    m = re.findall(r"####\s*([^\n]+)", text)
    return m[-1].strip() if m else None


def extract_boxed_answer(text: str) -> Optional[str]:
    """Extract the answer inside \\boxed{...} (MATH benchmark convention)."""
    m = re.findall(r"\\boxed\{([^}]*)\}", text)
    return m[-1].strip() if m else None


def normalize_answer_string(ans: Optional[str]) -> str:
    """
    Normalise answer strings for string-equality comparison.
    Strips whitespace, dollar signs, commas, and lowercases.
    NOTE: This does NOT handle LaTeX equivalences (e.g., \\frac{1}{2} ≠ 0.5).
    For the paper, note this limitation; the MATH benchmark ideally needs
    sympy-based normalisation for full accuracy.
    """
    if ans is None:
        return ""
    ans = ans.strip().replace("$", "").replace(",", "").replace(" ", "")
    return ans.lower()


def extract_option_letter(text: str) -> Optional[str]:
    """
    FIX-10: Extract the FIRST option letter (A-D) from a model response.

    The original notebook returned the LAST match. But models typically output
    the answer letter first ("A. The answer is A because…"), and taking the
    last match would pick up trailing letters in the explanation.

    We also try to match just at the start of the output first for robustness.
    """
    text = text.strip()
    # Prefer a letter at the very start of the response
    m = re.match(r"^\s*([A-E])\b", text)
    if m:
        return m.group(1)
    # Fall back to first letter found anywhere
    m = re.findall(r"\b([A-E])\b", text)
    return m[0] if m else None

In [27]:
def generate_text(
    model: nn.Module,
    tokenizer,
    prompt: str,
    cfg: RunConfig,
) -> str:
    """
    Generate a single completion from `prompt`.
    Called separately for each of the sampling_k samples in maj@1 evaluation.
    """
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_source_len,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=cfg.do_sample,
            temperature=cfg.temperature,
            top_p=cfg.top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the NEW tokens (exclude the input prompt tokens)
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

In [28]:
def evaluate_gsm8k(
    model: nn.Module, tokenizer, cfg: RunConfig, paths: Dict[str, Path]
) -> Dict[str, Any]:
    """
    Evaluate on the full GSM8K test set (1319 problems) using majority vote
    over sampling_k=5 independent samples per question.

    maj@1 = fraction of questions where the majority-vote answer is correct.
    This metric is more stable than greedy decoding for small models.
    """
    ds = normalize_gsm8k(load_gsm8k_dataset(), tokenizer)["test"]
    if cfg.max_eval_samples is not None:
        ds = ds.select(range(min(cfg.max_eval_samples, len(ds))))

    correct, records = 0, []

    for ex in tqdm(ds, desc="GSM8K eval"):
        votes = []
        for _ in range(cfg.sampling_k):
            text = generate_text(model, tokenizer, ex["prompt"], cfg)
            pred = normalize_answer_string(extract_hash_answer(text))
            votes.append(pred)

        majority = max(set(votes), key=votes.count) if votes else ""
        gold = normalize_answer_string(extract_hash_answer(ex["raw_answer"]))
        ok = int(majority == gold and gold != "")
        correct += ok
        records.append({
            "question": ex["raw_question"],
            "gold": gold,
            "pred_votes": votes,
            "pred_majority": majority,
            "correct": ok,
        })

    maj1 = correct / len(ds)
    save_json({"records": records, "maj1": maj1}, paths["evals"] / "gsm8k_eval.json")
    print(f"[GSM8K] maj@1 = {maj1:.4f}")
    return {"gsm8k_maj1": maj1}

In [29]:
def evaluate_math(
    model: nn.Module, tokenizer, cfg: RunConfig, paths: Dict[str, Path]
) -> Dict[str, Any]:
    """
    Evaluate on the MATH benchmark test set.
    Gold answers are in \\boxed{} format; predictions use #### if present,
    otherwise \\boxed{}.
    NOTE: String equality is used here. True MATH evaluation should use the
    sympy-based equivalence checker from the MATH repository.
    """
    ds = normalize_math_eval(load_math_dataset())["test"]
    if cfg.max_eval_samples is not None:
        ds = ds.select(range(min(cfg.max_eval_samples, len(ds))))

    correct, records = 0, []

    for ex in tqdm(ds, desc="MATH eval"):
        text = generate_text(model, tokenizer, ex["prompt"], cfg)
        pred = extract_hash_answer(text) or extract_boxed_answer(text)
        gold = extract_boxed_answer(ex["raw_answer"]) or extract_hash_answer(ex["raw_answer"])

        pred_n = normalize_answer_string(pred)
        gold_n = normalize_answer_string(gold)
        ok = int(pred_n == gold_n and gold_n != "")
        correct += ok
        records.append({"question": ex["raw_question"], "gold": gold_n,
                        "pred": pred_n, "correct": ok})

    acc = correct / len(ds)
    save_json({"records": records, "math_accuracy": acc},
              paths["evals"] / "math_eval.json")
    print(f"[MATH] accuracy = {acc:.4f}")
    return {"math_accuracy": acc}

In [ ]:
import copy

def extract_option_letter(text: str) -> Optional[str]:
    m = re.findall(r"\b([A-D])\b", text.strip())
    if m:
        return m[-1]
    return None


def evaluate_mmlu_5shot(
    model,
    tokenizer,
    cfg: RunConfig,
    max_subjects: Optional[int] = None,
    max_samples_per_subject: Optional[int] = None,
):
    """
    MMLU should be cheap in smoke/screen mode:
    - deterministic decoding
    - max_new_tokens very small
    - optional subject/sample caps
    """
    raw = load_mmlu_dataset()
    subjects = sorted(set(raw["test"]["subject"]))

    if max_subjects is not None:
        subjects = subjects[:max_subjects]

    subject_scores = {}
    records = []
    letters = ["A", "B", "C", "D"]

    # Use a local deterministic config for MCQ evaluation
    local_cfg = copy.deepcopy(cfg)
    local_cfg.do_sample = False
    local_cfg.temperature = 0.0
    local_cfg.top_p = 1.0
    local_cfg.max_new_tokens = 4
    local_cfg.num_beams = 1

    for subject in tqdm(subjects, desc="MMLU subjects"):
        subject_test = raw["test"].filter(lambda x: x["subject"] == subject)
        subject_dev = raw["dev"].filter(lambda x: x["subject"] == subject) if "dev" in raw else None

        fewshots = []
        if subject_dev is not None:
            for i in range(min(5, len(subject_dev))):
                fewshots.append({
                    "question": subject_dev[i]["question"],
                    "choices": subject_dev[i]["choices"],
                    "answer": subject_dev[i]["answer"],
                })

        if max_samples_per_subject is not None:
            subject_test = subject_test.select(range(min(max_samples_per_subject, len(subject_test))))

        correct = 0
        for ex in subject_test:
            prompt = build_mmlu_prompt(ex["question"], ex["choices"], fewshots)
            text = generate_text(model, tokenizer, prompt, local_cfg)
            pred_letter = extract_option_letter(text)
            gold_letter = letters[ex["answer"]]
            ok = int(pred_letter == gold_letter)
            correct += ok

            records.append({
                "subject": subject,
                "question": ex["question"],
                "pred": pred_letter,
                "gold": gold_letter,
                "correct": ok,
            })

        subject_scores[subject] = correct / max(len(subject_test), 1)

    overall = float(np.mean(list(subject_scores.values()))) if len(subject_scores) else 0.0
    save_json(
        {
            "subject_scores": subject_scores,
            "records": records,
            "mmlu_5shot_acc": overall,
            "num_subjects": len(subjects),
            "max_samples_per_subject": max_samples_per_subject,
        },
        paths["evals"] / "mmlu_eval.json",
    )
    return {"mmlu_5shot_acc": overall}

In [ ]:
import copy

def evaluate_arc(model, tokenizer, cfg: RunConfig) -> Dict[str, Any]:
    """
    ARC is also multiple-choice, so use short deterministic decoding.
    """
    ds = load_arc_dataset()["test"]
    if cfg.max_eval_samples is not None:
        ds = ds.select(range(min(cfg.max_eval_samples, len(ds))))

    local_cfg = copy.deepcopy(cfg)
    local_cfg.do_sample = False
    local_cfg.temperature = 0.0
    local_cfg.top_p = 1.0
    local_cfg.max_new_tokens = 4
    local_cfg.num_beams = 1

    correct = 0
    records = []

    for ex in tqdm(ds, desc="ARC eval"):
        question = ex["question"]
        texts = ex["choices"]["text"]
        prompt = build_arc_prompt(question, texts)

        out = generate_text(model, tokenizer, prompt, local_cfg)
        pred = extract_option_letter(out)
        gold = ex["answerKey"]

        ok = int(pred == gold)
        correct += ok
        records.append({
            "question": question,
            "pred": pred,
            "gold": gold,
            "correct": ok,
        })

    acc = correct / len(ds)
    save_json({"records": records, "arc_challenge_acc": acc}, paths["evals"] / "arc_eval.json")
    return {"arc_challenge_acc": acc}

In [32]:
def evaluate_mtbench_dump(
    model: nn.Module, tokenizer, cfg: RunConfig,
    paths: Dict[str, Path], run_id: str,
) -> Dict[str, Any]:
    """
    Generate answers for all 80 MT-Bench questions and write a JSONL file
    compatible with the FastChat judge pipeline.

    After running this, score with GPT-4 via:
      python -m fastchat.llm_judge.gen_judgment \
             --model-list <your_model_id> \
             --parallel 4

    The questions file path can be set via the MTBENCH_QUESTIONS_PATH env var.
    If not found, this evaluator is silently skipped.
    """
    q_path = os.environ.get(
        "MTBENCH_QUESTIONS_PATH",
        "mt_bench_questions/question.jsonl"
    )
    if not os.path.exists(q_path):
        print(f"[MT-Bench] Question file not found at {q_path}. Skipping.")
        print("           Set env var MTBENCH_QUESTIONS_PATH to enable.")
        return {"mtbench_dump_ready": 0}

    questions = []
    with open(q_path, "r") as f:
        for line in f:
            questions.append(json.loads(line.strip()))

    answers = []
    for q in tqdm(questions, desc="MT-Bench generation"):
        turns = q.get("turns", [])
        conv, turn_outputs = [], []

        for turn in turns:
            conv.append({"role": "user", "content": turn})
            try:
                prompt = tokenizer.apply_chat_template(
                    conv, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                prompt = turn + "\n\nAssistant:"

            reply = generate_text(model, tokenizer, prompt, cfg)
            turn_outputs.append(reply)
            conv.append({"role": "assistant", "content": reply})

        answers.append({
            "question_id": q.get("question_id", q.get("id")),
            "model_id": f"{cfg.method}-{run_id}",
            "choices": [{"index": 0, "turns": turn_outputs}],
            "tstamp": time.time(),
        })

    out_path = paths["preds"] / "mtbench_answers.jsonl"
    with open(out_path, "w") as f:
        for row in answers:
            f.write(json.dumps(row) + "\n")

    print(f"[MT-Bench] Answers written to {out_path}")
    return {"mtbench_dump_ready": 1, "mtbench_answer_file": str(out_path)}

In [33]:
def run_selected_evals(
    model: nn.Module, tokenizer, cfg: RunConfig,
    paths: Dict[str, Path], run_id: str,
) -> Dict[str, Any]:
    """
    Run only the eval buckets requested in cfg.eval_buckets.
    Each evaluator is wrapped in try/except so one failure doesn't abort
    the others — errors are printed and the metric is omitted from the row.
    """
    metrics = {}

    if "gsm8k" in cfg.eval_buckets:
        try:
            metrics.update(evaluate_gsm8k(model, tokenizer, cfg, paths))
        except Exception as e:
            print(f"[WARN] GSM8K eval failed: {e}")

    if "math" in cfg.eval_buckets:
        try:
            metrics.update(evaluate_math(model, tokenizer, cfg, paths))
        except Exception as e:
            print(f"[WARN] MATH eval failed: {e}")

    if "mmlu" in cfg.eval_buckets:
        try:
            metrics.update(evaluate_mmlu_5shot(
                model, tokenizer, cfg, paths,
                # Production: remove these caps
                max_subjects=None,
                max_samples_per_subject=None,
            ))
        except Exception as e:
            print(f"[WARN] MMLU eval failed: {e}")

    if "arc" in cfg.eval_buckets:
        try:
            metrics.update(evaluate_arc(model, tokenizer, cfg, paths))
        except Exception as e:
            print(f"[WARN] ARC eval failed: {e}")

    if "mtbench" in cfg.eval_buckets:
        try:
            metrics.update(evaluate_mtbench_dump(
                model, tokenizer, cfg, paths, run_id
            ))
        except Exception as e:
            print(f"[WARN] MT-Bench dump failed: {e}")

    return metrics

In [34]:
def train_model(
    cfg: RunConfig,
    tokenizer,
    train_loader: DataLoader,  # FIX-12: explicit argument, not global
) -> Tuple[nn.Module, Dict[str, Any], float, float]:
    """
    Unified training loop for all methods.

    Key design decisions:
    - FIX-1: AutoMode accumulates gradients across ALL micro-steps via
             AutoModeGradAccumulator, not just the last micro-step.
    - FIX-12: train_loader is passed explicitly to avoid hidden global state.
    - FIX-13: running_loss tracks the UNSCALED loss for accurate reporting.
    - AdaLoRA: model.update_and_allocate(global_step) is called every
               optimizer step as required by the PEFT library.
    - dyn_full_only: same switching logic as AutoMode but uses a completely
                     different switch function (set_dyn_full_only_layer).
    """
    # ── Setup ────────────────────────────────────────────────────────────────
    set_seed(cfg.seed)
    run_id = make_run_id(cfg)
    paths = ensure_dirs(cfg, run_id)
    save_json(asdict(cfg), paths["configs"] / "run_config.json")

    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    start_time = time.time()

    # Number of OPTIMIZER steps per epoch (micro-steps ÷ grad_accum)
    steps_per_epoch = math.ceil(len(train_loader) / cfg.grad_accum_steps)
    total_steps = steps_per_epoch * cfg.epochs

    # ── Build model ──────────────────────────────────────────────────────────
    model = build_model_for_method(cfg, tokenizer, train_loader)

    # AdaLoRA needs to be rebuilt now that we know total_steps
    if cfg.method == "adalora":
        # Re-apply with correct total_steps (the first call used dummy value)
        model = apply_adalora(model, cfg, total_steps)

    # ── Optimizer and scheduler ──────────────────────────────────────────────
    optimizer, scheduler = rebuild_optimizer_and_scheduler(
        model, cfg, total_steps, current_step=0
    )

    # ── AutoMode / dyn_full_only state ───────────────────────────────────────
    grad_accum = AutoModeGradAccumulator()
    dynamic_history = []
    is_dynamic = cfg.method in ("automode", "dyn_full_only")

    if is_dynamic:
        # How often (in optimizer steps) to check layer importance
        update_interval = max(1, steps_per_epoch // max(cfg.dynamic_updates, 1))
        print(f"[{cfg.method}] Switching every {update_interval} optimizer steps")
    else:
        update_interval = None

    # ── Training ─────────────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad(set_to_none=True)
    global_step = 0
    running_loss = 0.0
    micro_step_in_accum = 0

    for epoch in range(cfg.epochs):
        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{cfg.epochs}")

        for batch in pbar:
            # Move batch to GPU
            batch = {k: v.to(model.device) for k, v in batch.items()
                     if isinstance(v, torch.Tensor)}

            # ── Forward + backward ───────────────────────────────────────────
            outputs = model(**batch)
            # Divide loss by grad_accum_steps so gradients are averaged
            # across the accumulation window (not summed).
            loss = outputs.loss / cfg.grad_accum_steps
            loss.backward()

            # FIX-13: accumulate UNSCALED loss for accurate logging
            running_loss += outputs.loss.item()
            micro_step_in_accum += 1

            # FIX-1: AutoMode gradient accumulation after EVERY backward pass
            if is_dynamic:
                grad_accum.accumulate(model)

            # ── Optimizer step (every grad_accum_steps micro-steps) ──────────
            if micro_step_in_accum == cfg.grad_accum_steps:
                # ── AutoMode / dyn_full_only switching ───────────────────────
                if is_dynamic and update_interval is not None:
                    if (global_step + 1) % update_interval == 0:
                        scores = grad_accum.get_scores_and_clear()

                        if scores:
                            vals = np.array(list(scores.values()))
                            threshold = float(np.percentile(vals, cfg.dynamic_threshold))
                            all_ids = get_all_transformer_layer_ids(model)

                            promoted = []
                            for idx in all_ids:
                                is_full_ft = (
                                    scores.get(idx, 0.0) >= threshold
                                )
                                if cfg.method == "automode":
                                    set_layer_mode(model, idx, is_full_ft)
                                else:  # dyn_full_only
                                    set_dyn_full_only_layer(model, idx, is_full_ft)
                                if is_full_ft:
                                    promoted.append(idx)

                            # Rebuild optimizer: param groups changed
                            optimizer, scheduler = rebuild_optimizer_and_scheduler(
                                model, cfg, total_steps, current_step=global_step
                            )

                            trainable, total = count_trainable_params(model)
                            dynamic_history.append({
                                "step": global_step + 1,
                                "threshold": threshold,
                                "promoted_to_full_ft": promoted,
                                "trainable_params": trainable,
                            })

                # ── AdaLoRA rank budget update ───────────────────────────────
                # MUST be called after backward but before optimizer step.
                if cfg.method == "adalora":
                    model.update_and_allocate(global_step)

                # ── Standard optimizer step ──────────────────────────────────
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), cfg.max_grad_norm
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                # Logging
                avg_loss = running_loss / cfg.grad_accum_steps
                pbar.set_postfix({"loss": f"{avg_loss:.4f}", "step": global_step})
                running_loss = 0.0
                micro_step_in_accum = 0
                global_step += 1

    # ── Post-training ─────────────────────────────────────────────────────────
    train_time_sec = time.time() - start_time
    peak_vram_gb = (
        torch.cuda.max_memory_allocated() / 1024**3
        if torch.cuda.is_available() else 0.0
    )

    if is_dynamic and dynamic_history:
        save_json(
            {"history": dynamic_history},
            paths["dynamic"] / "dynamic_layers.json"
        )

    if cfg.save_model:
        save_dir = paths["checkpoints"] / "final_model"
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"[INFO] Model saved to {save_dir}")

    print(
        f"[INFO] Training complete — {train_time_sec / 60:.1f} min, "
        f"peak VRAM {peak_vram_gb:.2f} GB"
    )

    # ── Evaluation ────────────────────────────────────────────────────────────
    metrics = run_selected_evals(model, tokenizer, cfg, paths, run_id)
    append_result_row(cfg, run_id, metrics, train_time_sec, peak_vram_gb)

    return model, metrics, train_time_sec, peak_vram_gb

In [35]:
# Quick way to construct RunConfig objects for common baselines.
# Override any field you need after construction.

def preset_full_ft(track="gsm8k", seed=8):
    return RunConfig(train_track=track, method="full_ft", seed=seed)

def preset_lora(track="gsm8k", r=16, alpha=32, seed=8):
    return RunConfig(train_track=track, method="lora",
                     lora_r=r, lora_alpha=alpha, seed=seed)

def preset_dora(track="gsm8k", r=16, alpha=32, seed=8):
    return RunConfig(train_track=track, method="dora",
                     lora_r=r, lora_alpha=alpha, seed=seed)

def preset_adalora(track="gsm8k", seed=8):
    return RunConfig(train_track=track, method="adalora", seed=seed)

def preset_loraga(track="gsm8k", r=16, alpha=32, seed=8):
    return RunConfig(train_track=track, method="loraga",
                     lora_r=r, lora_alpha=alpha, seed=seed)

def preset_topk(track="gsm8k", start=13, end=19, seed=8):
    return RunConfig(train_track=track, method="topk_deep_block",
                     deep_block_start=start, deep_block_end=end, seed=seed)

def preset_automode(track="gsm8k", r=16, alpha=32, u=6, t=10, seed=8):
    """
    Best AutoMode config from 9B experiments: u=6 (6 switches/epoch), t=10
    (10th percentile threshold → top 90% of layers by gradient norm get FFT).
    """
    return RunConfig(
        train_track=track, method="automode",
        lora_r=r, lora_alpha=alpha,
        dynamic_updates=u, dynamic_threshold=t,
        seed=seed,
    )

def preset_dyn_full_only(track="gsm8k", u=6, t=10, seed=8):
    """
    AutoMode ablation: same controller, but no LoRA fallback for non-selected
    layers. Proves whether the LoRA channel is the key contribution.
    """
    return RunConfig(
        train_track=track, method="dyn_full_only",
        dynamic_updates=u, dynamic_threshold=t,
        seed=seed,
    )

In [36]:
def run_one(cfg: RunConfig):
    """
    End-to-end: build tokenizer + data → train → eval → log.
    Returns (model, metrics, train_time_sec, peak_vram_gb).
    """
    print(f"\n{'=' * 60}")
    print(f"  Method: {cfg.method}  |  Track: {cfg.train_track}  |  Seed: {cfg.seed}")
    print(f"{'=' * 60}")

    tokenizer = get_tokenizer(cfg.model_name)

    # Load and tokenise training data
    train_ds = load_training_track(cfg, tokenizer)
    train_key = "train" if "train" in train_ds else list(train_ds.keys())[0]
    train_tokenized = build_train_dataset(train_ds[train_key], tokenizer, cfg)

    collator = CausalLMCollator(tokenizer)
    train_loader = DataLoader(
        train_tokenized,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        collate_fn=collator,
        num_workers=0,      # 0 is safer with CUDA tensors
        pin_memory=False,
    )

    return train_model(cfg, tokenizer, train_loader)

In [38]:
if __name__ == "__main__":
    # ── Smoke test: AutoMode on GSM8K, seed 8 ──────────────────────────────
    cfg = preset_automode(track="gsm8k", r=16, alpha=32, u=6, t=10, seed=8)
    cfg.max_train_samples = 200    # remove for full runs
    cfg.max_eval_samples = 50      # remove for full runs

    model, metrics, t_sec, vram = run_one(cfg)
    print("\nMetrics:", metrics)
    print(f"Time: {t_sec/60:.1f} min  |  Peak VRAM: {vram:.2f} GB")


  Method: automode  |  Track: gsm8k  |  Seed: 8


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[automode] Trainable: 3,194,880 / 2,617,536,768 (0.122%)
[automode] Switching every 2 optimizer steps


Epoch 2/2: 100%|██████████| 200/200 [00:43<00:00,  4.59it/s, loss=0.9467, step=24]


[INFO] Model saved to runs_gemma2_2b/fb42aacc17da/checkpoints/final_model
[INFO] Training complete — 1.5 min, peak VRAM 11.79 GB


...

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

all/auxiliary_train-00000-of-00001.parqu(…):   0%|          | 0.00/47.5M [00:00<?, ?B/s]

MMLU subjects:   4%|▎         | 2/57 [44:33<20:25:26, 1336.84s/it]


KeyboardInterrupt: 

In [ ]:
def run_screen_grid(track="gsm8k", seed=8):
    """
    Run all methods on the chosen track with one seed.
    Good for a quick sanity check before launching the full 3-seed grid.
    """
    screen_cfgs = [
        preset_full_ft(track=track, seed=seed),
        preset_lora(track=track, r=16, seed=seed),
        preset_lora(track=track, r=32, seed=seed),
        preset_lora(track=track, r=64, seed=seed),
        preset_dora(track=track, r=16, seed=seed),
        preset_adalora(track=track, seed=seed),
        preset_loraga(track=track, r=16, seed=seed),
        preset_topk(track=track, seed=seed),
        preset_automode(track=track, u=6, t=10, seed=seed),
        preset_dyn_full_only(track=track, u=6, t=10, seed=seed),
    ]

    all_results = []
    for c in screen_cfgs:
        _, metrics, t_sec, vram = run_one(c)
        all_results.append({
            "method": c.method,
            "seed": c.seed,
            **metrics,
            "train_time_sec": round(t_sec, 1),
            "peak_vram_gb": round(vram, 2),
        })
        gc.collect()
        torch.cuda.empty_cache()

    df = pd.DataFrame(all_results)
    print("\n" + df.to_string(index=False))
    return df

In [ ]:
def run_production_grid(track="gsm8k"):
    """
    Run all headline methods across 3 seeds — this is what goes in Table 1.
    For the full NeurIPS paper, also call this with track="metamathqa" and
    track="alpaca_gpt4".
    """
    SEEDS = [8, 25, 42]
    HEADLINE_METHODS = [
        preset_full_ft,
        lambda track, seed: preset_lora(track=track, r=16, seed=seed),
        lambda track, seed: preset_dora(track=track, r=16, seed=seed),
        preset_adalora,
        lambda track, seed: preset_topk(track=track, seed=seed),
        lambda track, seed: preset_automode(track=track, u=6, t=10, seed=seed),
        lambda track, seed: preset_dyn_full_only(track=track, u=6, t=10, seed=seed),
    ]

    for method_fn in HEADLINE_METHODS:
        for seed in SEEDS:
            cfg = method_fn(track=track, seed=seed)
            try:
                run_one(cfg)
            except Exception as e:
                print(f"[ERROR] {cfg.method} seed={seed} failed: {e}")
            gc.collect()
            torch.cuda.empty_cache()